# Codex Python SDK Playground
This notebook is ready for experimenting with the app-server Python SDK.

## 0) One-time install (run in terminal, not notebook)
```bash
python3 -m venv .venv-notebook
source .venv-notebook/bin/activate
pip install -U pip
pip install "git+https://github.com/Shadimrad/codex-py-sdk.git@python-sdk-polish#subdirectory=sdk/python"
pip install jupyter
# make sure codex is installed + logged in
codex --version
codex login
```

In [ ]:
from codex_app_server import AppServerClient
print("SDK import ok")

## 1) Quick conversation

In [ ]:
with AppServerClient() as client:
    client.initialize()
    thread = client.thread_start_session(model="gpt-5")
    result = thread.ask_result("Give me 3 concise bullets on gradient descent.")
    print("Thread:", result.thread_id)
    print(result.text)

## 2) Stream text chunks

In [ ]:
with AppServerClient() as client:
    client.initialize()
    thread = client.thread_start_session()
    for chunk in thread.stream_text("Write a short haiku about compilers."):
        print(chunk, end="", flush=True)
    print()

## 3) Typed + schema wrappers

In [ ]:
with AppServerClient() as client:
    client.initialize()

    started_typed = client.thread_start_typed(model="gpt-5")
    print("typed thread id:", started_typed.thread.id)

    started_schema = client.thread_start_schema(model="gpt-5")
    print("schema thread id:", started_schema.thread.id)

## 4) Parse notifications

In [ ]:
with AppServerClient() as client:
    client.initialize()
    thread = client.thread_start_session()
    turn = thread.turn_text("Say hello in one sentence.")
    turn_id = turn["turn"]["id"]
    while True:
        n = client.next_notification()
        typed = client.parse_notification_typed(n)
        if typed is not None:
            print("typed:", type(typed).__name__)
        if n.method == "turn/completed" and (n.params or {}).get("turn",{}).get("id") == turn_id:
            break

## 5) Async usage

In [ ]:
import asyncio
from codex_app_server import AsyncAppServerClient

async def main():
    async with AsyncAppServerClient() as client:
        await client.initialize()
        thread = await client.thread_start_session(model="gpt-5")
        result = await thread.ask_result("Give me 2 quick facts about SIMD.")
        print(result.text)

asyncio.run(main())